# Boundary Accuracy Summary Table

## What question this answers

What fraction of algorithm boundaries land exactly on GT, within 1 frame, within 5,
within 10, or beyond 10 — and how many boundaries are phantom (no GT match) or
missed (GT with no algo match)? This is a single-snapshot accuracy profile.

## Column definitions (NON-overlapping buckets)

The percentage columns partition ALL algorithm+GT boundaries into exhaustive,
mutually exclusive buckets. They MUST sum to 100% for each row.

| Column | Definition |
|--------|------------|
| N_GT | Number of ground-truth boundaries |
| N_algo | Number of algorithm-emitted boundaries |
| delta=0 (%) | Matched boundaries with exactly 0 frame error |
| abs(delta)=1 (%) | Matched boundaries with exactly 1 frame error |
| 2<=abs(delta)<=5 (%) | Matched boundaries with 2-5 frames error |
| 6<=abs(delta)<=10 (%) | Matched boundaries with 6-10 frames error |
| abs(delta)>10 (%) | Matched boundaries with >10 frames error |
| miss (%) | GT boundaries with no algo match |
| phantom (%) | Algo boundaries with no GT match |
| median abs(delta) | Median absolute error (matched only, frames) |
| mean signed delta | Mean signed error (matched only, frames) |

## What improvement looks like

- delta=0 fraction increases.
- Tail buckets (6-10, >10) shrink toward 0.
- phantom and miss reach 0.
- median abs(delta) decreases.
- mean signed delta approaches 0.

## Red-flag patterns

- Buckets not summing to 100% means a counting bug.
- High phantom+miss means the algorithm is finding the wrong number of segments.
- Large mean signed delta means systematic directional bias.

In [ ]:
# === PARAMS ===
from pathlib import Path

SNAPSHOT_DIR = Path(r"Y:\2_Connectome\Behavior\MouseReach_Pipeline\Improvement_Snapshots\segmentation\seg_v2.2.0_multi_proposer")
FIGSIZE = (16, 4)
DPI = 300
SAVE = False  # Flip to True to write PNG + legend

In [ ]:
# === LOAD ===
import json
import pandas as pd
import numpy as np

deltas_path = SNAPSHOT_DIR / "metrics" / "boundary_deltas.csv"
scalars_path = SNAPSHOT_DIR / "metrics" / "scalars.json"

df_all = pd.read_csv(deltas_path)
with open(scalars_path, "r") as f:
    scalars = json.load(f)

df_matched = df_all[df_all["status"] == "matched"].copy()
df_matched["signed_delta"] = df_matched["signed_delta"].astype(int)
df_matched["abs_delta"] = df_matched["signed_delta"].abs()

print(f"Loaded {len(df_all)} rows, {len(df_matched)} matched")

In [ ]:
# === COMPUTE ===
from mousereach.improvement.lib.palette import (
    SEGMENTATION_LABELS, SEGMENTATION_SUBSET_ORDER,
)

rows = []

for key in SEGMENTATION_SUBSET_ORDER:
    sc = scalars[key] if key != "all" else scalars["all"]
    if key == "all":
        matched = df_matched
        full = df_all
    else:
        matched = df_matched[df_matched["subset_tag"] == key]
        full = df_all[df_all["subset_tag"] == key]

    n_gt = int(sc["n_gt_boundaries"])
    n_algo = int(sc["n_algo_boundaries"])
    n_phantom = int(sc["n_phantom"])
    n_miss = int(sc["n_miss"])
    n_matched = len(matched)

    # Denominator: all events = matched + phantom + miss
    total = n_matched + n_phantom + n_miss

    abs_d = matched["abs_delta"].values if len(matched) > 0 else np.array([])

    # NON-overlapping buckets (matched)
    n_d0 = int((abs_d == 0).sum())
    n_d1 = int((abs_d == 1).sum())
    n_d2_5 = int(((abs_d >= 2) & (abs_d <= 5)).sum())
    n_d6_10 = int(((abs_d >= 6) & (abs_d <= 10)).sum())
    n_d_gt10 = int((abs_d > 10).sum())

    # Sanity: matched buckets must sum to n_matched
    assert n_d0 + n_d1 + n_d2_5 + n_d6_10 + n_d_gt10 == n_matched, (
        f"Bucket sum mismatch for {key}: "
        f"{n_d0}+{n_d1}+{n_d2_5}+{n_d6_10}+{n_d_gt10} != {n_matched}"
    )

    # Percentages (of total)
    def pct(n):
        return round(100 * n / total, 1) if total > 0 else 0.0

    pct_d0 = pct(n_d0)
    pct_d1 = pct(n_d1)
    pct_d2_5 = pct(n_d2_5)
    pct_d6_10 = pct(n_d6_10)
    pct_d_gt10 = pct(n_d_gt10)
    pct_miss = pct(n_miss)
    pct_phantom = pct(n_phantom)

    # Assert 100% sum
    pct_sum = pct_d0 + pct_d1 + pct_d2_5 + pct_d6_10 + pct_d_gt10 + pct_miss + pct_phantom
    assert abs(pct_sum - 100.0) < 0.5, (
        f"Percentage sum for {key} = {pct_sum:.1f}%, expected ~100%"
    )

    # Summary stats (matched only)
    median_abs = float(np.median(abs_d)) if len(abs_d) > 0 else float("nan")
    mean_signed = float(np.mean(matched["signed_delta"].values)) if len(matched) > 0 else float("nan")

    rows.append({
        "Subset": SEGMENTATION_LABELS[key],
        "N_GT": n_gt,
        "N_algo": n_algo,
        "delta=0 (%)": pct_d0,
        "|delta|=1 (%)": pct_d1,
        "2-5 (%)": pct_d2_5,
        "6-10 (%)": pct_d6_10,
        ">10 (%)": pct_d_gt10,
        "miss (%)": pct_miss,
        "phantom (%)": pct_phantom,
        "med|d|": median_abs,
        "mean d": round(mean_signed, 2),
    })

table_df = pd.DataFrame(rows)
print(table_df.to_string(index=False))
print("\n100% sum assertion: PASSED")

In [ ]:
# === RENDER ===
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

fig, ax = plt.subplots(figsize=FIGSIZE, dpi=DPI)
ax.axis("off")

col_labels = list(table_df.columns)
cell_text = []
cell_colors = []

# Color scale for accuracy cells: green (good) to white (neutral) to light red (bad)
# delta=0 is best when HIGH, tail buckets best when LOW
good_color = "#C8E6C9"   # Light green
neutral_color = "#FFFFFF" # White
bad_color = "#FFCDD2"    # Light red
header_color = "#E8EAF6"  # Light indigo

# Columns that are "good when high"
good_high_cols = {"delta=0 (%)"}
# Columns that are "good when low" (bad when high)
good_low_cols = {"2-5 (%)", "6-10 (%)", ">10 (%)", "miss (%)", "phantom (%)"}
# Neutral columns (no coloring)
neutral_cols = {"Subset", "N_GT", "N_algo", "|delta|=1 (%)", "med|d|", "mean d"}

for _, row in table_df.iterrows():
    row_text = []
    row_colors = []
    for col in col_labels:
        val = row[col]
        if isinstance(val, float):
            if col in ("med|d|", "mean d"):
                row_text.append(f"{val:.1f}" if not (val != val) else "--")
            else:
                row_text.append(f"{val:.1f}")
        else:
            row_text.append(str(val))

        # Coloring
        if col in good_high_cols:
            # Interpolate: 0% = neutral, 100% = fully green
            frac = min(float(val) / 100.0, 1.0) if isinstance(val, (int, float)) else 0
            r = int(255 - frac * (255 - 200))
            g = int(255 - frac * (255 - 230))
            b = int(255 - frac * (255 - 201))
            row_colors.append(f"#{r:02X}{g:02X}{b:02X}")
        elif col in good_low_cols:
            # Interpolate: 0% = green, high% = red
            frac = min(float(val) / 20.0, 1.0) if isinstance(val, (int, float)) else 0
            if frac < 0.01:
                row_colors.append(good_color)
            else:
                r = int(200 + frac * (255 - 200))
                g = int(230 - frac * (230 - 205))
                b = int(201 - frac * (201 - 210))
                row_colors.append(f"#{r:02X}{g:02X}{b:02X}")
        else:
            row_colors.append(neutral_color)

    cell_text.append(row_text)
    cell_colors.append(row_colors)

table = ax.table(
    cellText=cell_text,
    colLabels=col_labels,
    cellColours=cell_colors,
    colColours=[header_color] * len(col_labels),
    loc="center",
    cellLoc="center",
)

table.auto_set_font_size(False)
table.set_fontsize(9)
table.scale(1, 1.6)

# Bold header
for (row_idx, col_idx), cell in table.get_celld().items():
    if row_idx == 0:
        cell.set_text_props(fontweight="bold", fontsize=8)
    cell.set_edgecolor("#CCCCCC")

ax.set_title("Boundary accuracy summary", fontsize=14, fontweight="bold", pad=20)

plt.tight_layout()
plt.show()

In [ ]:
# === SAVE ===
if SAVE:
    fig_dir = SNAPSHOT_DIR / "figures"
    fig_dir.mkdir(parents=True, exist_ok=True)

    png_path = fig_dir / "summary_table.png"
    fig.savefig(str(png_path), dpi=DPI, bbox_inches="tight", pad_inches=0.15,
                facecolor="white")
    print(f"Saved: {png_path}")

    legend_md = f"""# Boundary Accuracy Summary Table

## What question this answers

What fraction of algorithm boundaries land exactly on GT, within 1 frame, within 5,
within 10, or beyond 10 -- and how many boundaries are phantom (no GT match) or
missed (GT with no algo match)?

## Column definitions (NON-overlapping buckets)

Percentage columns partition ALL boundaries into exhaustive, mutually exclusive buckets
that sum to 100%.

| Column | Definition |
|--------|------------|
| N_GT | Number of ground-truth boundaries |
| N_algo | Number of algorithm-emitted boundaries |
| delta=0 (%) | Matched with exactly 0 frame error |
| abs(delta)=1 (%) | Matched with exactly 1 frame error |
| 2-5 (%) | Matched with 2-5 frames error |
| 6-10 (%) | Matched with 6-10 frames error |
| >10 (%) | Matched with >10 frames error |
| miss (%) | GT boundaries with no algo match |
| phantom (%) | Algo boundaries with no GT match |
| median abs(delta) | Median absolute error (matched only) |
| mean signed delta | Mean signed error (matched only) |

## Rendering params

- SNAPSHOT_DIR: `{SNAPSHOT_DIR}`
- FIGSIZE: {FIGSIZE}
- DPI: {DPI}
"""
    legend_path = fig_dir / "summary_table_legend.md"
    legend_path.write_text(legend_md, encoding="utf-8")
    print(f"Saved: {legend_path}")
else:
    print("SAVE=False -- set SAVE=True to write PNG + legend")